# Projeto: Normalização e Preparação dos Dados Meteorológicos – Clima Tempo

## Descrição Geral
Este notebook trata da normalização, padronização e persistência dos dados brutos da fonte **Clima Tempo**, originalmente localizados na camada *bronze*.  
Antes de serem processados neste notebook, os dados passam por uma etapa automatizada de **validação com Great Expectations (GX)**. Apenas os registros **validados com sucesso** são utilizados nas transformações abaixo.  


## Objetivos

- Padronizar os nomes das colunas conforme convenção definida.
- Realizar o ajuste de tipos de dados para colunas de datas, timestamps e decimal.
- Separar colunas compostas (como precipitação e probabilidade).
- Preparar o DataFrame para persistência na camada silver.

## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake

## Fonte de Dados

- **Localização:** [Tabela Clima Tempo](grupo01.bronze.clima_tempo)
- **Formato:** Delta
- **Origem:** Tabela bronze criada a partir da ingestão do dataset Clima Tempo

---

### Histórico de Atualizações

| Data       | Responsável  | Versão | Descrição                                               |
|------------|-------------|--------|---------------------------------------------------------|
| 16/05/2025 | Lucas Berta | 1.0    | Pipeline de ingestão dos dados da fonte Clima Tempo    |
| 16/05/2025 | Lucas Berta | 1.1    | Criação da tabela silver para persistência dos dados   |
| 19/05/2025 | Evelise Dias | 1.2    | Atualização de documentação   |
| 25/05/2025 | Evelise Dias   | 1.3    | Ajuste do pipeline para ler apenas dados validados pelo GX                     |
| 26/05/2025 | Evelise Dias   | 1.4    | Atualização da documentação para refletir a origem validada dos dados          |




In [0]:
%run ../utils/utils_run_silver

In [0]:
%run ../../src/silver/queries_silver

In [0]:
pasta_validos = lakehouse_logs_gx + "/bronze/clima_tempo/validos/clima_tempo"
df_clima_tempo = ler_tabelas(pasta_validos, fromTable=False)

In [0]:
df_clima_renomeado = renomear_colunas(df_clima_tempo, CONVENCAO_COLUNAS_CLIMA_TEMPO)

In [0]:
df_transformado = alterar_tipos(df_clima_renomeado, ["dt_previsao"], "date")

In [0]:
df_transformado = tratar_celulas(df_transformado)

In [0]:
df_transformado = alterar_tipos(df = df_transformado,colunas= ["umid_min", "umid_max", "temp_min", "temp_max", "precipitacao", "vel_vento", "prob_precipitacao"], tipo= "decimal")

In [0]:
df_transformado = alterar_tipos(df = df_transformado,colunas= ["dt_carga"], tipo= "timestamp", data_formato="yyyy-MM-dd HH:mm:ss")

In [0]:
spark.sql(create_table_silver_clima_tempo)

In [0]:
df_com_id = gera_primary_key(df_transformado)

In [0]:
df_sem_duplicados = remover_duplicatas(df_com_id, ["dt_carga", "dt_previsao", "sig_uf","nom_municipio", "flg_fonte_previsao"])

In [0]:
inserir_com_merge(dataframe = df_sem_duplicados, tabela_destino=caminho_tabela_clima_tempo_silver, camposChave=["id"])